## Evaluation - Part 2

In this notebook, we focus our efforts in evaluating how to best link to occupations. In the previous one, we found that linking titles to embeddings corresponding to a combination of all fields yielded better results than linking sentences describing the job to either combination of preferred label, secondary labels and description.

The second step includes the following questions:

1. If we allow multiple embeddings to correspond to a single node in the ESCO database, do we get a better recall?
2. What is the lowest value of k for which we get a recall of 1 (or, for lack of a better result, a sufficiently high value?)

We will define some preliminary functions and start with the first question:

In [1]:
# 1. Loading the test dataset for occupations using the Huggingface library
from huggingface_hub import hf_hub_download
import pandas as pd
from tqdm import tqdm
import os 
from dotenv import load_dotenv
from vertexai.language_models import TextEmbeddingModel

tqdm.pandas()

load_dotenv()

HF_TOKEN = os.environ["HF_ACCESS_TOKEN"]
OCCUPATION_REPO_ID = "tabiya/hahu_test"
OCCUPATION_FILENAME = "redacted_hahu_test_with_id.csv"
OCCUPATION_DATA_PATH = "https://raw.githubusercontent.com/tabiya-tech/taxonomy-model-application/main/data-sets/csv/tabiya-esco-v1.1.1/occupations.csv"


df_occupation_test = pd.read_csv(
    hf_hub_download(repo_id=OCCUPATION_REPO_ID, filename=OCCUPATION_FILENAME, repo_type="dataset", token=HF_TOKEN)
)
df_occupation_database = pd.read_csv(OCCUPATION_DATA_PATH)

model = TextEmbeddingModel.from_pretrained("textembedding-gecko@003")

/Users/francescopreta/miniconda3/envs/backend/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from typing import Any, Dict, List, Optional, Tuple

def precision_at_k(prediction: List[List[str]], true: List[List[str]], k: Optional[int] = None):
    """Calculates the average precision at k considering
    for each prediction the number of correct retrieved nodes
    divided by the number of total retrieved nodes.

    Args:
        prediction (List[List[str]]): list of 
            predicted lists, each with the corresponding
            nodes.
        true (List[List[str]]): list of the multiple true nodes 
            for each sample in the dataset.
        k (Optional[int]): number of samples of the prediction to consider.
            When None considers all the elements of the list.

    Returns:
        float: average precision at k over all the test set.
    """
    assert len(prediction) == len(true)
    total_precision = 0
    for pred_list, true_val in zip(prediction, true):
        if k:
            pred_list = pred_list[:k]
            tot_samples = k
        else:
            tot_samples = len(pred_list)
        total_precision+=len(set(pred_list).intersection(set(true_val)))/tot_samples
    return total_precision/len(true)

def recall_at_k(prediction: List[List[str]], true: List[List[str]], k: Optional[int] = None):
    """Calculates the average recall at k considering
    for each prediction the number of correct retrieved nodes
    divided by the number of total correct nodes.

    Args:
        prediction (List[List[str]]): list of 
            predicted lists, each with the corresponding
            nodes.
        true (List[List[str]]): list of the multiple true nodes 
            for each sample in the dataset.
        k (Optional[int]): number of predicted samples to consider.
            When None considers all of them. Defaults to None.

    Returns:
        float: average recall at k over all the test set.
    """
    assert len(prediction) == len(true)
    total_recall = 0
    for pred_list, true_val in zip(prediction, true):
        if k:
            pred_list = pred_list[:k]
        total_recall+=len(set(pred_list).intersection(set(true_val)))/len(set(true_val))
    return total_recall/len(true)

def f_score(prec: float, rec: float) -> float:
    """Returns the f-score corresponding to
    a given precision and recall.

    Args:
        prec (float): provided precision
        rec (float): provided recall

    Returns:
        float: resulting f-score
    """
    return 2*prec*rec/(prec+rec)

def get_all_metrics(predictions: List[List[str]], true_values: List[List[str]], k: Optional[int]=None) -> Tuple[float,float,float]:
    """Get recall, precision and F-score for given results and
    true values.

    Args:
        predictions (List[List[str]]): list of predictions.
        true_values (List[List[str]]): list of true values.
        k (Optional[int]): number of predicted samples to consider.
            When None considers all of them. Defaults to None.

    Returns:
        Tuple[float,float,float]: recall, precision and F-score.
    """
    rec_at_k = recall_at_k(predictions, true_values, k)
    prec_at_k = precision_at_k(predictions, true_values, k)
    f_score_at_k = f_score(prec_at_k, rec_at_k)
    return rec_at_k, prec_at_k, f_score_at_k

In [3]:
# Write a function to embed strings in batches
from typing import List

def embed_strings_in_batch(list_of_queries: List[str], model: TextEmbeddingModel, batch_size: int = 100) -> List[List[float]]:
    """Uses a TextEmbeddingModel to embed a list of queries in batches.

    Args:
        list_of_queries (List[str]): list of queries to be embedded in batches.
        model (TextEmbeddingModel): embedding model.
        batch_size (int, optional): size of each batch which should be less than or equal to 250.
            Defaults to 100.

    Returns:
        List[List[float]]: List of embeddings corresponding to the queries.
    """
    assert batch_size<=250
    embedding_results = []
    num_samples = len(list_of_queries)
    for i in range(int(num_samples/batch_size)+1):
        batch = list_of_queries[i*batch_size:(i+1)*batch_size]
        embedding_results += model.get_embeddings(batch)
    assert len(embedding_results) == len(list_of_queries)
    return [embedding_result.values for embedding_result in embedding_results]

### 1. Does segmentation through fields improve recall on Occupation?

In our previous experiment we wanted to find the best method to link the occupations of the test set to the ESCO database. We allowed each ESCO node to have a single embedding that could only be a combination of the strings in its fields (preferred label, secondary labels and description). We found that for low values of k, the preferred label guaranteed a higher recall, while higher values of k a combination of all fields guaranteed a better outcome.

We now want to consider the possibility that each node can be represented by multiple embeddings. This is similar to the way in which documents can be segmented and embedded in information retrieval. We consider two approaches:

1. We embed each document with three embeddings: preferred label, secondary labels and description. 
2. We embed each document with more than three embeddings, including one embedding for each secondary label.

We first generate the results and then compare their recall to the results of the previous evaluation. We compare the results for both synthetic queries and titles.

Notice that in our retrieval function, we consider k to be the set of unique top esco codes within the first 100 entries, meaning that the more embeddings per node we load in the collection, the least amount of different nodes we will find.

In [4]:
import chromadb

client = chromadb.Client()

def create_collections(db_name: str, df_database: pd.DataFrame):
    """Creates multiple collections for each choice of db_name
    and corresponding documents and embeddings.

    Args:
        db_name (str): name of the database. 
            Either 'skills' or 'occupations'.
        df_database (pd.DataFrame): database containing the metadata.
    """
    collection = client.create_collection(name=db_name)
    batch_number = int(len(df_database)/41655)+1
    for i in range(batch_number):
        temp_database = df_database.iloc[i*41655:(i+1)*41655]
        collection.add(
            documents = list(temp_database["text"]),
            metadatas = [{"CODE": row["CODE"]} for _, row in temp_database.iterrows()],
            embeddings = list(temp_database["embedding"]),
            ids = [f"id_{i*41655+j}" for j in range(len(temp_database))]
        )

In [5]:
# Generate a dataframe having three rows for each node including the possible fields
from tqdm import tqdm 

df_occupation_full_list = []
for _, row in tqdm(df_occupation_database.iterrows()):
    for field in ["PREFERREDLABEL", "ALTLABELS", "DESCRIPTION"]:
        df_occupation_full_list.append({"text": str(row[field]), "CODE":row["CODE"]})
df_occupation_full_df = pd.DataFrame(df_occupation_full_list)
df_occupation_full_df["embedding"] = embed_strings_in_batch(list(df_occupation_full_df["text"]), model)

df_occupation_test["CODE"] = df_occupation_test["esco_code"].apply(lambda x: [x])

create_collections("occupation_three_embeddings", df_occupation_full_df)

3007it [00:00, 57404.18it/s]


In [6]:
from tqdm import tqdm 

df_occupation_full_list = []
for _, row in tqdm(df_occupation_database.iterrows()):
    for field in ["PREFERREDLABEL", "DESCRIPTION"]:
        df_occupation_full_list.append({"text": row[field], "CODE":row["CODE"]})
    for sec_label in str(row["ALTLABELS"]).split("\n"):
        df_occupation_full_list.append({"text": sec_label, "CODE":row["CODE"]})
df_occupation_full_df = pd.DataFrame(df_occupation_full_list)
df_occupation_full_df["embedding"] = embed_strings_in_batch(list(df_occupation_full_df["text"]), model)

df_occupation_test["CODE"] = df_occupation_test["esco_code"].apply(lambda x: [x])

create_collections("occupation_multiple_embeddings", df_occupation_full_df)

3007it [00:00, 39254.86it/s]


In [7]:
def get_results_from_embeddings(embeddings: List[List[float]], collection: chromadb.Collection, n_results:int=100) -> Tuple[List[List[str]], List[List[str]]]:
    """Utility function to return results of embedding queries
    to a given collection.

    Args:
        embeddings (List[List[float]]): List of embeddings for queries.
        collection (chromadb.Collection): ChromaDB collection to query.
        n_results (int): number of results to retrieve from the collection

    Returns:
        Tuple[List[List[str]], List[List[str]]]: List of results, one for
            regular vector search, the other one for maximal marginal relevance
            search. Each element is a list of string corresponding to the
            search result for the embedding in the same position in the input list. 
    """
    vector_search_results = []
    for embedding in embeddings:
        # Find the top 100 documents and save them in vector_search_results
        documents_from_search = collection.query(query_embeddings=embedding, n_results=n_results, include=["metadatas", "embeddings"])
        vector_search_results.append([elem["CODE"] for elem in documents_from_search["metadatas"][0]])
    return vector_search_results


def run_eval(collection_name: str, df_test: pd.DataFrame, target_column: str = "CODE", embedding_column = "embedding") -> pd.DataFrame:
    """Returns the results of an evaluation on a list of collections

    Args:
        collection_name (str): name of the collection.
        df_test (pd.DataFrame): test dataframe, containing an embedding column
            and a test_target column.

    Returns:
        pd.DataFrame: dataframe with the result of the evaluation depending on the
            different hyperparameters.
    """
    eval_data = []
    # Fetch collection
    collection = client.get_collection(collection_name)
    # Initialize lists to save results
    vector_search_results = get_results_from_embeddings(list(df_test[embedding_column]), collection)
    # Evaluate accuracy at k for k=1, 3, 5, 10
    single_vs_results = [list(set(elem)) for elem in vector_search_results]
    vector_search_results_single = [sorted(elem, key=vs_elem.index) for elem, vs_elem in zip(single_vs_results, vector_search_results)]
    for k in [1, 3, 5, 10]:
        rec_at_k, prec_at_k, f_score_at_k = get_all_metrics(vector_search_results_single, list(df_test[target_column]), k)
        eval_data.append({"method": collection_name, "embedded field": "title" if embedding_column=="embedding_title" else "synthetic query", "k":k, "recall": round(rec_at_k, 4), "precision": round(prec_at_k,4), "f-score": round(f_score_at_k,4)})
# Save the results in a dataframe
    eval_df = pd.DataFrame(eval_data)
    return eval_df

In [8]:
# Embed the titles and synthetic queries in the test set

df_occupation_test["embedding"] = embed_strings_in_batch(list(df_occupation_test["synthetic_query"]), model)
df_occupation_test["embedding_title"] = embed_strings_in_batch(list(df_occupation_test["title"]), model)

KeyboardInterrupt: 

In [10]:
# Run the full evaluation for titles and synthetic queries
eval_df = pd.DataFrame()
for collection_name in ["occupation_multiple_embeddings", "occupation_three_embeddings"]:
    eval_df = pd.concat([eval_df, run_eval(collection_name, df_occupation_test)])
    eval_df = pd.concat([eval_df, run_eval(collection_name, df_occupation_test, embedding_column="embedding_title")])
    


The results are as follows, also including those from the previous evaluation on occupation

| Method                          | k  | Recall | Precision | F-score | Input Type       |
|---------------------------------|----|--------|-----------|---------|------------------|
| occupation_multiple_embeddings | 10 | 0.7657 | 0.0766    | 0.1392  | title            |
| occupation_three_embeddings    | 10 | 0.7601 | 0.076     | 0.1382  | synthetic query  |
| occupation_three_embeddings    | 10 | 0.7546 | 0.0755    | 0.1372  | title            |
| ALL_OCCUPATIONS                 | 10 | 0.7491 | 0.0749    | 0.1362  | title            |
| occupation_multiple_embeddings | 10 | 0.7435 | 0.0744    | 0.1352  | synthetic query  |
| ALL_OCCUPATIONS                 | 10 | 0.738  | 0.0738    | 0.1342  | synthetic query |
| occupation_three_embeddings    | 5  | 0.6919 | 0.1384    | 0.2306  | title            |
| occupation_three_embeddings    | 5  | 0.6919 | 0.1384    | 0.2306  | synthetic query  |
| ALL_OCCUPATIONS                 | 5  | 0.6882 | 0.1376    | 0.2294  | title            |
| ALL_OCCUPATIONS                 | 5  | 0.6605 | 0.1321    | 0.2202  | synthetic query  |
| occupation_multiple_embeddings | 5  | 0.6568 | 0.1314    | 0.2189  | title            |
| occupation_multiple_embeddings | 5  | 0.6494 | 0.1299    | 0.2165  | synthetic query  |
| occupation_three_embeddings    | 3  | 0.6421 | 0.214     | 0.321   | title            |
| ALL_OCCUPATIONS                 | 3  | 0.6162 | 0.2054    | 0.3081  | title            |
| occupation_three_embeddings    | 3  | 0.5959 | 0.1986    | 0.298   | synthetic query  |
| occupation_multiple_embeddings | 3  | 0.583  | 0.1943    | 0.2915  | title            |
| ALL_OCCUPATIONS                 | 3  | 0.5812 | 0.1937    | 0.2906  | synthetic query  |
| occupation_multiple_embeddings | 3  | 0.5646 | 0.1882    | 0.2823  | synthetic query  |
| occupation_three_embeddings    | 1  | 0.441  | 0.441     | 0.441   | title            |
| ALL_OCCUPATIONS                 | 1  | 0.4262 | 0.4262    | 0.4262  | title            |
| occupation_three_embeddings    | 1  | 0.3819 | 0.3819    | 0.3819  | synthetic query  |
| occupation_multiple_embeddings | 1  | 0.3579 | 0.3579    | 0.3579  | synthetic query  |
| occupation_multiple_embeddings | 1  | 0.3561 | 0.3561    | 0.3561  | title            |
| ALL_OCCUPATIONS                 | 1  | 0.3469 | 0.3469    | 0.3469  | synthetic query  |

We can notice the following trends:

- When using the **synthetic query**, embedding preferred labels, secondary labels and description separately is stably more effective than embedding them all together or than separating all the possible secondary labels. This makes sense, as we get some advantages over the multiple secondary labels embeddings by avoiding flooding the database with the same embedding for multiple labels. This is also stably better than the combination of all fields, probably because the embeddings are more focused and the more descriptive queries can be matched to the description directly, while those that are focused on the title can be matched to either preferred or secondary labels.
- When using the **title**, the method with three embeddings is more effective than all the others for low values of k. For higher values of k this doesn't hold anymore and the method with multiple embeddings has some gains as the titles are probably closer to each single preferred label. Since we don't care that most of the labels are not correct, this guarantees an advantage over the flooding of labels.

### 2. For which value of k do we obtain a sufficiently high recall value?

Using the collections defined in the previous section, we now turn to the question of how many occupations should we iterate over if we want to find the correct job. The simplified assumption in this case is that every sample has exactly one correct ESCO node, while this might not be the case in theory. However, by getting a value when the true positive is only one per example gives us an important higher bound on how many occupations are needed to find the correct one in most cases.

In what follows, we would like to aim for a 100% recall. However, since this is not realistic in our experimental setting, we decide to aim for the highest recall such that the search doesn't repeatedly find any additional data as we increase k. The highest recall in this sense is the one that can stay constant for 100 values of k.

In [9]:
def get_highest_recall(
        collection: chromadb.Collection, 
        df_test: pd.DataFrame,
        embedding_column: str,
        target_column: str,
        n_results: int =100
        ) -> int:
    """Function to find the lowest value of k for which we get
    maximized recall (that is, the recall doesn't change for any
    larger k up to 100 higher).

    Args:
        collection (chromadb.Collection): collection from which
            we retrieve the occupation.
        df_test (pd.DataFrame): test dataframe with codes to retrieve.
        embedding_column (str): name of the embedding column in the test
            dataframe.
        target_column (str): name of the target column in the test dataframe.
        n_results (int): number of results to retrieve from the collection.

    Returns:
        Tuple[int, float]: lowest value of k giving the highest recall and
            highest value of recall
    """
    # Find vector search results
    vector_search_results = get_results_from_embeddings(list(df_test[embedding_column]), collection, n_results)
    # Find and order only one result per ESCO code
    single_vs_results = [list(set(elem)) for elem in vector_search_results]
    vector_search_results_single = [sorted(elem, key=vs_elem.index) for elem, vs_elem in zip(single_vs_results, vector_search_results)]
    # Define the lowest value of k and its recall
    k=1
    rec_at_k=0
    # Counter that will be reset everytime the recall improves
    improving_counter = 100
    while improving_counter>0 and rec_at_k<0.99:
        # Calculate recall at k
        temp_rec_at_k = recall_at_k(vector_search_results_single, list(df_test[target_column]), k)
        # If it's the same as before, reduce the counter
        if temp_rec_at_k<=rec_at_k:
            improving_counter-=1
        # Else reset the counter
        else:
            improving_counter = 100
        # Update recall and k
        rec_at_k = temp_rec_at_k
        k+=1
    if improving_counter==0:
        return k-100, rec_at_k
    return k, rec_at_k



We now evaluate the k and the recall for the two collections we have available

In [10]:
top_k, top_recall = get_highest_recall(client.get_collection("occupation_three_embeddings"), df_occupation_test, "embedding", "CODE")
print(f"For the collection with three embeddings per ESCO node, we get a maximum recall of {round(top_recall, 2)} at k={top_k}")
top_k, top_recall = get_highest_recall(client.get_collection("occupation_multiple_embeddings"), df_occupation_test, "embedding", "CODE")
print(f"For the collection with more than three embeddings per ESCO node, we get a maximum recall of {round(top_recall, 2)} at k={top_k}")


For the collection with three embeddings per ESCO node, we get a maximum recall of 0.9 at k=82
For the collection with more than three embeddings per ESCO node, we get a maximum recall of 0.88 at k=42


Interestingly enough, there is a trade-off between using more embeddings (thus having more chances of finding the right node) and flooding the number of entries for a single node. In our case, we get higher maximum recalls using only three embeddings, but we get the higher value earlier if we use more than three embeddings. This happens because we retrieve only a limited amount of embeddings at every iteration, so that the ranking is not done over the whole ESCO dataset. We can increase the number of embeddings we retrieve and observe how this trade-off is even more marked:

In [11]:
for n_results in [200, 500, 1000]:
    top_k, top_recall = get_highest_recall(client.get_collection("occupation_three_embeddings"), df_occupation_test, "embedding", "CODE", n_results)
    print(f"Within the first {n_results} results:")
    print(f"For the collection with three embeddings per ESCO node, we get a maximum recall of {round(top_recall, 2)} at k={top_k}")
    top_k, top_recall = get_highest_recall(client.get_collection("occupation_multiple_embeddings"), df_occupation_test, "embedding", "CODE", n_results)
    print(f"For the collection with more than three embeddings per ESCO node, we get a maximum recall of {round(top_recall, 2)} at k={top_k}")


Within the first 200 results:
For the collection with three embeddings per ESCO node, we get a maximum recall of 0.94 at k=145
For the collection with more than three embeddings per ESCO node, we get a maximum recall of 0.91 at k=79
Within the first 500 results:
For the collection with three embeddings per ESCO node, we get a maximum recall of 0.98 at k=379
For the collection with more than three embeddings per ESCO node, we get a maximum recall of 0.96 at k=165
Within the first 1000 results:
For the collection with three embeddings per ESCO node, we get a maximum recall of 0.99 at k=573
For the collection with more than three embeddings per ESCO node, we get a maximum recall of 0.98 at k=312


###  3. Evaluating localised ESCO datasets

Since we would like to be able to connect users and companies with skills and occupations that are specific to a given country, we now analyze how our method applies to localised data in the example of South Africa. In order to do so, we load a test set containing 1549 SMS in which participants answered to a question about their livelihood and day to day job. These questions are then linked to the localised South Africa ESCO database.

The localization process of a given database simply adds secondary labels to existing leaf nodes of the standard ESCO database. No new nodes are added and no other fields than secondary labels are changed.

In [3]:
import pandas as pd

sa_test_df = pd.read_csv("/Users/francescopreta/coding/compass/backend/esco_search/_scripts/sa_test_set.csv")
sa_occupation_database_df = pd.read_csv("/Users/francescopreta/coding/compass/backend/esco_search/_scripts/esco_occupations_fc_220424.csv")


In our evaluation, we want to understand a few things:

1. Since localised data is concentrated in the secondary labels, which embedding methods guarantees the highest recall? Should we separate the secondary labels, or keep them together?
2. How much is the mapping to the localised ESCO database better than the mapping to the traditional one when it comes to recall in the localised test set?
3. When restricting to the subset of the test set which is localised (that is, the ESCO codes whose secondary label is different from the standard), which method performs best?

We start by creating two databases in ChromaDB that contain localised data in a similar fashion to the first experiment of this notebook. After calculating the embeddings and regularizing the data, we then evaluate the accuracy on the localised test set for linking to these new databases as well as the old ones.

In [11]:
# Generate a localised dataframe having three rows for each node including the possible fields
from tqdm import tqdm 

df_occupation_sa_list = []
for _, row in tqdm(sa_occupation_database_df.iterrows()):
    for field in ["PREFERREDLABEL", "ALTLABELS", "DESCRIPTION"]:
        df_occupation_sa_list.append({"text": str(row[field]), "CODE":row["CODE"]})
df_occupation_sa = pd.DataFrame(df_occupation_sa_list)
df_occupation_sa["embedding"] = embed_strings_in_batch(list(df_occupation_sa["text"]), model)

create_collections("sa_occupation_three_embeddings", df_occupation_sa)

3007it [00:00, 58244.81it/s]


In [12]:
# Generate a localised dataframe having multiple rows for each node including the possible fields
# as well as all the separated secondary labels

df_occupation_sa_all_list = []
for _, row in tqdm(sa_occupation_database_df.iterrows()):
    for field in ["PREFERREDLABEL", "DESCRIPTION"]:
        df_occupation_sa_all_list.append({"text": row[field], "CODE":row["CODE"]})
    for sec_label in str(row["ALTLABELS"]).split("\n"):
        df_occupation_sa_all_list.append({"text": sec_label, "CODE":row["CODE"]})
df_occupation_sa_all = pd.DataFrame(df_occupation_sa_all_list)
df_occupation_sa_all["embedding"] = embed_strings_in_batch(list(df_occupation_sa_all["text"]), model)

create_collections("sa_occupation_multiple_embeddings", df_occupation_full_df)

In [15]:
# Embed and regularise test data

sa_test_df["embedding"] = embed_strings_in_batch(list(sa_test_df["Text"]), model)
sa_test_df["CODE"] = sa_test_df["Esco Code"].apply(lambda x: [str(x)])

In [16]:
# Run the evaluation to standard and localised databases

eval_df = pd.DataFrame()
for collection_name in ["occupation_multiple_embeddings", "occupation_three_embeddings", "sa_occupation_multiple_embeddings", "sa_occupation_three_embeddings"]:
    eval_df = pd.concat([eval_df, run_eval(collection_name, sa_test_df)])    

The results of the evaluation are as follows:

| method                              | k   | recall | precision | f-score |
|-------------------------------------|-----|--------|-----------|---------|
| occupation_three_embeddings         | 10  | 0.4551 | 0.0455    | 0.0828  |
| sa_occupation_three_embeddings      | 10  | 0.4532 | 0.0453    | 0.0824  |
| sa_occupation_multiple_embeddings   | 10  | 0.4513 | 0.0451    | 0.082   |
| occupation_multiple_embeddings      | 10  | 0.4041 | 0.0404    | 0.0735  |
| occupation_three_embeddings         | 5   | 0.366  | 0.0732    | 0.122   |
| sa_occupation_three_embeddings      | 5   | 0.3635 | 0.0727    | 0.1212  |
| sa_occupation_multiple_embeddings   | 5   | 0.3402 | 0.068     | 0.1134  |
| occupation_multiple_embeddings      | 5   | 0.3338 | 0.0668    | 0.1113  |
| sa_occupation_three_embeddings      | 3   | 0.3015 | 0.1005    | 0.1507  |
| occupation_three_embeddings         | 3   | 0.2989 | 0.0996    | 0.1495  |
| sa_occupation_multiple_embeddings   | 3   | 0.2699 | 0.09      | 0.1349  |
| occupation_multiple_embeddings      | 3   | 0.2666 | 0.0889    | 0.1333  |
| occupation_three_embeddings         | 1   | 0.1827 | 0.1827    | 0.1827  |
| sa_occupation_three_embeddings      | 1   | 0.1821 | 0.1821    | 0.1821  |
| occupation_multiple_embeddings      | 1   | 0.1646 | 0.1646    | 0.1646  |
| sa_occupation_multiple_embeddings   | 1   | 0.1472 | 0.1472    | 0.1472  |

As we can see, the method with three embeddings outperforms the one in which the labels are distinct, both when using the localised version of the database and when using the non-localised. As in the previous example for the standard version, this advantage decreases the larger the k, which is to be expected.

Moreover, at the current state, the non-localised version of ESCO is working quite well in identifying the correct node. This might happen for a few reasons, that could be further investigated:
1. The test set links mostly to nodes that are not localised.
2. The test set links to nodes that are localised, but the text doesn't refer to the localised secondary labels.
3. The methods considered (embeddings or segmentation) don't perform well on localised data. 

As we observe in the next iteration, most of the test set links to localised nodes, so the reason is definitely not the first one.

In [6]:
# Find the subset of the test set that links to localised nodes
localised_node_list = []
for _, row in sa_occupation_database_df.iterrows():
    original_node = df_occupation_database[df_occupation_database["CODE"]==row["CODE"]]
    if len(original_node) == 1:
        original_node = original_node.iloc[0]
        if original_node["DESCRIPTION"]!=row["DESCRIPTION"] or original_node["PREFERREDLABEL"]!=row["PREFERREDLABEL"] or original_node["ALTLABELS"]!=row["ALTLABELS"]:
            localised_node_list.append(row["CODE"])
    elif len(original_node) > 1:
        raise Exception   

localised_sa_test_df = sa_test_df[sa_test_df["Esco Code"].apply(str).isin(localised_node_list)]
print(len(localised_sa_test_df))

1450


In [21]:
# Evaluate linking to the localised dataframe

localised_eval_df = pd.DataFrame()
for collection_name in ["occupation_multiple_embeddings", "occupation_three_embeddings","sa_occupation_multiple_embeddings", "sa_occupation_three_embeddings"]:
    localised_eval_df = pd.concat([localised_eval_df, run_eval(collection_name, localised_sa_test_df)])    

The results of the evaluation are as follows:

| method                              | k   | recall | precision | f-score |
|-------------------------------------|-----|--------|-----------|---------|
| occupation_three_embeddings         | 10  | 0.4393 | 0.0439    | 0.0799  |
| sa_occupation_multiple_embeddings   | 10  | 0.4393 | 0.0439    | 0.0799  |
| sa_occupation_three_embeddings      | 10  | 0.4372 | 0.0437    | 0.0795  |
| occupation_multiple_embeddings      | 10  | 0.3883 | 0.0388    | 0.0706  |
| occupation_three_embeddings         | 5   | 0.3476 | 0.0695    | 0.1159  |
| sa_occupation_three_embeddings      | 5   | 0.3448 | 0.069     | 0.1149  |
| sa_occupation_multiple_embeddings   | 5   | 0.3283 | 0.0657    | 0.1094  |
| occupation_multiple_embeddings      | 5   | 0.32   | 0.064     | 0.1067  |
| sa_occupation_three_embeddings      | 3   | 0.2814 | 0.0938    | 0.1407  |
| occupation_three_embeddings         | 3   | 0.2786 | 0.0929    | 0.1393  |
| sa_occupation_multiple_embeddings   | 3   | 0.2614 | 0.0871    | 0.1307  |
| occupation_multiple_embeddings      | 3   | 0.2538 | 0.0846    | 0.1269  |
| occupation_three_embeddings         | 1   | 0.169  | 0.169     | 0.169   |
| sa_occupation_three_embeddings      | 1   | 0.1683 | 0.1683    | 0.1683  |
| occupation_multiple_embeddings      | 1   | 0.1559 | 0.1559    | 0.1559  |
| sa_occupation_multiple_embeddings   | 1   | 0.1386 | 0.1386    | 0.1386  |

Since the size of the dataset is not changed much from the previous iteration, we can find the same conclusions. Further work will need to be conducted to establish whether the test set refers explicitly to the secondary labels and in case whether there are more effective methods to highlight the localised secondary labels during the search. Additionally, we should discuss with researchers the semantic relationship between the new secondary labels and the requests, highlighting which samples in the test sets refer explicitly to such labels. Finally, we should wait for an updated version of the localised South African ESCO database.